In [ ]:
import pandas as pd
import numpy as np
import math
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import os
from google.colab import drive

drive.mount('/content/drive')

# Load main malaria dataset
!cp "/content/drive/MyDrive/hasil_data_gabungan.csv" "./data.csv"
df = pd.read_csv("data.csv")

# Clean numeric columns
for col in ["Kasus Malaria", "Curah Hujan", "Kelembapan", "Suhu"]:
    df[col] = df[col].astype(str).str.replace(",", "").str.strip()
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.dropna(inplace=True)

# Load DMI & ONI data
!cp "/content/drive/MyDrive/DATA/ONI (Tahun).xlsx" "./ONI_3.xlsx"
!cp "/content/drive/MyDrive/DATA/DMI pertahun (Tahun).xlsx" "./DMI_pertahun.xlsx"
dmi_df = pd.read_excel("DMI_pertahun.xlsx")
oni_df = pd.read_excel("ONI_3.xlsx")

# Merge DMI and ONI into main dataframe
df = df.merge(dmi_df, on="Tahun", how="left")
df = df.merge(oni_df, on="Tahun", how="left")
df.dropna(inplace=True)

# Sequence preparation
def create_sequences(data, n_steps=2):
    X, y = [], []
    for i in range(len(data) - n_steps):
        X.append(data[i:i+n_steps, 1:])  # input: all but Kasus
        y.append(data[i+n_steps, 0])     # target: Kasus Malaria
    return np.array(X), np.array(y)

def inverse_scale(preds, scaler):
    dummy = np.zeros((len(preds), 5))  # 5 dummy features to pad to 6
    return scaler.inverse_transform(np.hstack((preds, dummy)))[:, 0]

# Config
n_steps = 2
performance_results = []
provinces = df["Nama Provinsi"].unique()

training_histories = {}

for province in provinces:
    print(f"\nProcessing: {province}")
    df_prov = df[df["Nama Provinsi"] == province].sort_values("Tahun").copy()

    if len(df_prov) < 5:
        print(f"Skipping {province} due to insufficient data.")
        continue

    # Normalize
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df_prov[["Kasus Malaria", "Curah Hujan", "Kelembapan", "Suhu", "DMI", "ONI"]])
    X, y = create_sequences(scaled, n_steps)
    if len(X) < 1:
        print(f"Skipping {province} due to insufficient sequences.")
        continue

    # Split
    split = int(len(X) * 0.8)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    # 1-Layer LSTM
    model = Sequential()
    model.add(LSTM(54, input_shape=(n_steps, X.shape[2])))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

    # Train
    early_stop = EarlyStopping(monitor='val_loss', patience=100, restore_best_weights=True)
    history = model.fit(X_train, y_train, epochs=2000, batch_size=1, verbose=0,
              callbacks=[early_stop], validation_data=(X_test, y_test))
    training_histories[province] = history.history

    # Predict
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    y_train_inv = inverse_scale(y_train.reshape(-1, 1), scaler)
    y_train_pred_inv = inverse_scale(y_train_pred, scaler)
    y_test_inv = inverse_scale(y_test.reshape(-1, 1), scaler)
    y_test_pred_inv = inverse_scale(y_test_pred, scaler)

    # Metrics
    train_rmse = math.sqrt(mean_squared_error(y_train_inv, y_train_pred_inv))
    train_mae = mean_absolute_error(y_train_inv, y_train_pred_inv)
    test_rmse = math.sqrt(mean_squared_error(y_test_inv, y_test_pred_inv))
    test_mae = mean_absolute_error(y_test_inv, y_test_pred_inv)
    train_r2 = r2_score(y_train_inv, y_train_pred_inv)
    test_r2 = r2_score(y_test_inv, y_test_pred_inv)

    performance_results.append({
        'Province': province,
        'Model': 'LSTM_1Layer_6Features',
        'Train RMSE': train_rmse,
        'Train MAE': train_mae,
        'Train R2': train_r2,
        'Test RMSE': test_rmse,
        'Test MAE': test_mae,
        'Test R2': test_r2
    })

# Save results
performance_df = pd.DataFrame(performance_results)
performance_df.to_csv("performance_results_1Layer.csv", index=False)
print("\nPerformance Summary:")
print(performance_df)


Mounted at /content/drive

Processing: ACEH


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step

Processing: SUMATERA UTARA


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step

Processing: SUMATERA BARAT


...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step

Processing: PAPUA BARAT


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step

Processing: PAPUA


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step

Performance Summary:
                Province                  Model    Train RMSE     Train MAE  \
0                   ACEH  LSTM_1Layer_6Features    837.135418    561.844593   
1         SUMATERA UTARA  LSTM_1Layer_6Features   3254.142058   2277.605959   
2         SUMATERA BARAT  LSTM_1Layer_6Features    657.501622    542.960525   
3                   RIAU  LSTM_1Layer_6Features    495.376646    434.336864   
4                  JAMBI  LSTM_1Layer_6Features   1756.781657   1465.662814   
5       SUMATERA SELATAN  LSTM_1Layer_6Features   1492.403716   1167.870877   
6               BENGKULU  LSTM_1Layer_6Features   3793.397119   2832.708208   
7                LAMPUNG  LSTM_1Layer_6Features   2307.128581   1975.187477   
8   KEP. BANGKA BELITUNG  LSTM_1Layer_6Features   1258.752547    860.282739   
9              KEP. RIAU  LSTM_1Layer_6Features   1561.995402    980.408537   
10           DKI JAKARTA  LSTM_

In [ ]:
import pandas as pd
import numpy as np
import math
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import os
from google.colab import drive

drive.mount('/content/drive')

# Load main malaria dataset
!cp "/content/drive/MyDrive/hasil_data_gabungan.csv" "./data.csv"
df = pd.read_csv("data.csv")

# Clean numeric columns
for col in ["Kasus Malaria", "Curah Hujan", "Kelembapan", "Suhu"]:
    df[col] = df[col].astype(str).str.replace(",", "").str.strip()
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.dropna(inplace=True)

# Load DMI & ONI data
!cp "/content/drive/MyDrive/DATA/ONI (Tahun).xlsx" "./ONI_3.xlsx"
!cp "/content/drive/MyDrive/DATA/DMI pertahun (Tahun).xlsx" "./DMI_pertahun.xlsx"
dmi_df = pd.read_excel("DMI_pertahun.xlsx")
oni_df = pd.read_excel("ONI_3.xlsx")

# Merge DMI and ONI into main dataframe
df = df.merge(dmi_df, on="Tahun", how="left")
df = df.merge(oni_df, on="Tahun", how="left")
df.dropna(inplace=True)

# Sequence preparation
def create_sequences(data, n_steps=2):
    X, y = [], []
    for i in range(len(data) - n_steps):
        X.append(data[i:i+n_steps, 1:])  # input: all but Kasus
        y.append(data[i+n_steps, 0])     # target: Kasus Malaria
    return np.array(X), np.array(y)

def inverse_scale(preds, scaler):
    dummy = np.zeros((len(preds), 5))  # 5 dummy features to pad to 6
    return scaler.inverse_transform(np.hstack((preds, dummy)))[:, 0]

# Config
n_steps = 2
performance_results = []
provinces = df["Nama Provinsi"].unique()

training_histories = {}

for province in provinces:
    print(f"\nProcessing: {province}")
    df_prov = df[df["Nama Provinsi"] == province].sort_values("Tahun").copy()

    if len(df_prov) < 5:
        print(f"Skipping {province} due to insufficient data.")
        continue

    # Normalize
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df_prov[["Kasus Malaria", "Curah Hujan", "Kelembapan", "Suhu", "DMI", "ONI"]])
    X, y = create_sequences(scaled, n_steps)
    if len(X) < 1:
        print(f"Skipping {province} due to insufficient sequences.")
        continue

    # Split
    split = int(len(X) * 0.8)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    # 1-Layer LSTM
    model = Sequential()
    model.add(LSTM(50, input_shape=(n_steps, X.shape[2])))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.06), loss='mse')

    # Train
    early_stop = EarlyStopping(monitor='val_loss', patience=100, restore_best_weights=True)
    history = model.fit(X_train, y_train, epochs=2000, batch_size=1, verbose=0,
              callbacks=[early_stop], validation_data=(X_test, y_test))
    training_histories[province] = history.history

    # Predict
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    y_train_inv = inverse_scale(y_train.reshape(-1, 1), scaler)
    y_train_pred_inv = inverse_scale(y_train_pred, scaler)
    y_test_inv = inverse_scale(y_test.reshape(-1, 1), scaler)
    y_test_pred_inv = inverse_scale(y_test_pred, scaler)

    # Metrics
    train_rmse = math.sqrt(mean_squared_error(y_train_inv, y_train_pred_inv))
    train_mae = mean_absolute_error(y_train_inv, y_train_pred_inv)
    test_rmse = math.sqrt(mean_squared_error(y_test_inv, y_test_pred_inv))
    test_mae = mean_absolute_error(y_test_inv, y_test_pred_inv)
    train_r2 = r2_score(y_train_inv, y_train_pred_inv)
    test_r2 = r2_score(y_test_inv, y_test_pred_inv)

    performance_results.append({
        'Province': province,
        'Model': 'LSTM_1Layer_6Features',
        'Train RMSE': train_rmse,
        'Train MAE': train_mae,
        'Train R2': train_r2,
        'Test RMSE': test_rmse,
        'Test MAE': test_mae,
        'Test R2': test_r2
    })

# Save results
performance_df = pd.DataFrame(performance_results)
performance_df.to_csv("performance_results_1Layer.csv", index=False)
print("\nPerformance Summary:")
print(performance_df)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Processing: ACEH


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 213ms/step

Processing: SUMATERA UTARA


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 210ms/step

Processing: SUMATERA BARAT


...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 200ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step

Processing: PAPUA BARAT


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 217ms/step

Processing: PAPUA


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step

Performance Summary:
                Province                  Model    Train RMSE     Train MAE  \
0                   ACEH  LSTM_1Layer_6Features    477.674024    370.823429   
1         SUMATERA UTARA  LSTM_1Layer_6Features   1017.374611    919.570914   
2         SUMATERA BARAT  LSTM_1Layer_6Features    407.264954    312.290691   
3                   RIAU  LSTM_1Layer_6Features    554.014750    493.615174   
4                  JAMBI  LSTM_1Layer_6Features   1855.512681   1372.158343   
5       SUMATERA SELATAN  LSTM_1Layer_6Features   1411.760912   1112.077056   
6               BENGKULU  LSTM_1Layer_6Features   2391.215332   2031.606363   
7                LAMPUNG  LSTM_1Layer_6Features   1858.362994   1668.891487   
8   KEP. BANGKA BELITUNG  LSTM_1Layer_6Features    985.982355    630.086975   
9              KEP. RIAU  LSTM_1Layer_6Features   1101.814058    535.106002   
10           DKI JAKARTA  LSTM_

In [ ]:
import pandas as pd
import numpy as np
import math
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import os
from google.colab import drive

drive.mount('/content/drive')

# Load main malaria dataset
!cp "/content/drive/MyDrive/hasil_data_gabungan.csv" "./data.csv"
df = pd.read_csv("data.csv")

# Clean numeric columns
for col in ["Kasus Malaria", "Curah Hujan", "Kelembapan", "Suhu"]:
    df[col] = df[col].astype(str).str.replace(",", "").str.strip()
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.dropna(inplace=True)

# Load DMI & ONI data
!cp "/content/drive/MyDrive/DATA/ONI (Tahun).xlsx" "./ONI_3.xlsx"
!cp "/content/drive/MyDrive/DATA/DMI pertahun (Tahun).xlsx" "./DMI_pertahun.xlsx"
dmi_df = pd.read_excel("DMI_pertahun.xlsx")
oni_df = pd.read_excel("ONI_3.xlsx")

# Merge DMI and ONI into main dataframe
df = df.merge(dmi_df, on="Tahun", how="left")
df = df.merge(oni_df, on="Tahun", how="left")
df.dropna(inplace=True)

# Sequence preparation
def create_sequences(data, n_steps=2):
    X, y = [], []
    for i in range(len(data) - n_steps):
        X.append(data[i:i+n_steps, 1:])  # input: all but Kasus
        y.append(data[i+n_steps, 0])     # target: Kasus Malaria
    return np.array(X), np.array(y)

def inverse_scale(preds, scaler):
    dummy = np.zeros((len(preds), 5))  # 5 dummy features to pad to 6
    return scaler.inverse_transform(np.hstack((preds, dummy)))[:, 0]

# Config
n_steps = 2
performance_results = []
provinces = df["Nama Provinsi"].unique()

training_histories = {}

for province in provinces:
  if province   == "DKI JAKARTA" or province == "PAPUA":
    print(f"\nProcessing: {province}")
    df_prov = df[df["Nama Provinsi"] == province].sort_values("Tahun").copy()

    if len(df_prov) < 5:
        print(f"Skipping {province} due to insufficient data.")
        continue

    # Normalize
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df_prov[["Kasus Malaria", "Curah Hujan", "Kelembapan", "Suhu", "DMI", "ONI"]])
    X, y = create_sequences(scaled, n_steps)
    if len(X) < 1:
        print(f"Skipping {province} due to insufficient sequences.")
        continue

    # Split
    split = int(len(X) * 0.8)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    # 1-Layer LSTM
    model = Sequential()
    model.add(LSTM(100, activation='relu', input_shape=(n_steps, X.shape[2])))
    model.add(Dropout(0.2))
    model.add(Dense(1, activation='linear'))
    model.compile(optimizer=Adam(learning_rate=0.1), loss='mse')

    # Train
    early_stop = EarlyStopping(monitor='val_loss', patience=500, restore_best_weights=True)
    history = model.fit(X_train, y_train, epochs=4000, batch_size=1, verbose=0,
              callbacks=[early_stop],
              validation_data=(X_test, y_test))
    training_histories[province] = history.history

    # Predict
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    y_train_inv = inverse_scale(y_train.reshape(-1, 1), scaler)
    y_train_pred_inv = inverse_scale(y_train_pred, scaler)
    y_test_inv = inverse_scale(y_test.reshape(-1, 1), scaler)
    y_test_pred_inv = inverse_scale(y_test_pred, scaler)

    # Metrics
    train_rmse = math.sqrt(mean_squared_error(y_train_inv, y_train_pred_inv))
    train_mae = mean_absolute_error(y_train_inv, y_train_pred_inv)
    test_rmse = math.sqrt(mean_squared_error(y_test_inv, y_test_pred_inv))
    test_mae = mean_absolute_error(y_test_inv, y_test_pred_inv)
    train_r2 = r2_score(y_train_inv, y_train_pred_inv)
    test_r2 = r2_score(y_test_inv, y_test_pred_inv)

    performance_results.append({
        'Province': province,
        'Model': 'LSTM_1Layer_6Features',
        'Train RMSE': train_rmse,
        'Train MAE': train_mae,
        'Train R2': train_r2,
        'Test RMSE': test_rmse,
        'Test MAE': test_mae,
        'Test R2': test_r2
    })

# Save results
performance_df = pd.DataFrame(performance_results)
performance_df.to_csv("performance_results_1Layer.csv", index=False)
print("\nPerformance Summary:")
print(performance_df)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Processing: DKI JAKARTA


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 215ms/step

Processing: PAPUA


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 212ms/step

Performance Summary:
      Province                  Model    Train RMSE     Train MAE  Train R2  \
0  DKI JAKARTA  LSTM_1Layer_6Features     49.010719     40.844381 -0.788489   
1        PAPUA  LSTM_1Layer_6Features  42245.552938  39546.181561 -0.056664   

      Test RMSE      Test MAE   Test R2  
0    113.657220     93.680114 -2.118774  
1  85218.747786  79976.122787 -2.071563  
